# Xanh Hub (Đề tài 6) – Huấn luyện & Triển khai Mô hình AI trên Google Colab GPU T4 + Google Drive

Notebook này phục vụ việc **fine-tune (QLoRA 4-bit)** mô hình ngôn ngữ mở nhẹ (`Qwen/Qwen2.5-1.5B-Instruct` / `Qwen2.5-3B-Instruct`, giấy phép Apache-2.0) cho hai bài toán của Xanh Hub:
1. **A1 (Trợ lý tạo ca bằng tiếng Việt tự nhiên):** Chuyển câu mô tả tiếng Việt của Điều phối viên thành cấu trúc JSON tạo ca trực.
2. **A4 (Giải thích phương án phân công bằng lời):** Diễn đạt kết quả so sánh thuật toán (B0, B1, X) và lý do thiếu người từ số liệu xác thực.

**Cách sử dụng:**
- Mở tệp này trên Google Colab, chọn `Runtime -> Change runtime type -> T4 GPU`.
- Chạy lần lượt các ô để gắn Google Drive, sinh dữ liệu huấn luyện, fine-tune LoRA và phát API qua **Cloudflare Tunnel** (`trycloudflare.com`) để ứng dụng PHP Xanh Hub ở máy cá nhân kết nối trực tiếp mà không tốn tài nguyên máy cục bộ.
- Khi không bật Colab (hoặc khi giảng viên chấm bài offline bằng `docker compose up`), `HybridAiService.php` tự động chuyển sang bộ phân tích quy tắc thuần PHP.

In [ ]:
# Bước 1: Kiểm tra GPU T4 và gắn kết Google Drive để lưu trọng số bền vững
!nvidia-smi
from google.colab import drive
import os

drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/XanhHub_AI_Colab_T4'
os.makedirs(SAVE_DIR, exist_ok=True)
print('Thư mục lưu mô hình trên Google Drive:', SAVE_DIR)

In [ ]:
# Bước 2: Sinh tập dữ liệu huấn luyện mẫu tiếng Việt chuẩn JSON Schema cho tính năng A1
import json

training_samples = [
    {
        "instruction": "Trích xuất thông tin tạo ca tình nguyện từ câu tiếng Việt sau ra JSON hợp lệ.",
        "input": "Cần 6 người dựng lều sáng thứ Bảy 7h-11h, tối thiểu 4 người, 2 người biết sơ cứu cấp 3",
        "output": json.dumps({
            "title": "Ca tình nguyện: Cần 6 người dựng lều sáng thứ Bảy 7h-11h",
            "day_of_week": 6,
            "date_ymd": "2026-11-07",
            "start_hm": "07:00",
            "end_hm": "11:00",
            "min_people": 4,
            "max_people": 6,
            "priority": 5,
            "requirements": [
                {"skill_code": "FIRST_AID", "skill_name": "Sơ cứu y tế", "min_level": 3, "required_count": 2},
                {"skill_code": "LOGISTICS", "skill_name": "Hậu cần & Dựng trại", "min_level": 2, "required_count": 4}
            ]
        }, ensure_ascii=False)
    },
    {
        "instruction": "Trích xuất thông tin tạo ca tình nguyện từ câu tiếng Việt sau ra JSON hợp lệ.",
        "input": "Tuyển 4 bạn truyền thông chụp ảnh lễ ra quân sáng Chủ Nhật 8h đến 11h30, tối thiểu 2 người cấp 3",
        "output": json.dumps({
            "title": "Ca tình nguyện: Truyền thông chụp ảnh lễ ra quân sáng Chủ Nhật",
            "day_of_week": 7,
            "date_ymd": "2026-11-08",
            "start_hm": "08:00",
            "end_hm": "11:30",
            "min_people": 2,
            "max_people": 4,
            "priority": 4,
            "requirements": [
                {"skill_code": "MEDIA", "skill_name": "Truyền thông & Ảnh", "min_level": 3, "required_count": 2}
            ]
        }, ensure_ascii=False)
    }
]

dataset_path = os.path.join(SAVE_DIR, 'xanhhub_a1_train.jsonl')
with open(dataset_path, 'w', encoding='utf-8') as f:
    for item in training_samples:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')
print('Đã lưu tập dữ liệu huấn luyện tại:', dataset_path)

In [ ]:
# Bước 3: Cài đặt thư viện QLoRA nhẹ và huấn luyện thử nghiệm trên GPU T4
!pip install -q transformers peft accelerate bitsandbytes datasets trl fastapi uvicorn
print('Đã cài đặt xong môi trường huấn luyện QLoRA trên Colab T4!')

In [ ]:
# Bước 4: Mở cổng API tương thích với HybridAiService.php qua Cloudflare Tunnel miễn phí
# Sao chép đường dẫn https://xxxx.trycloudflare.com/api/generate dán vào biến AI_ENDPOINT_URL trong tệp .env
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
print('Sẵn sàng phát API từ Colab T4 về ứng dụng PHP Xanh Hub!')